In [32]:
import pandas as pd
df_people = pd.read_csv('C:/Users/human/미정프로젝트/서울시 상권분석서비스(길단위인구-상권).csv', encoding='cp949')
df_area = pd.read_csv('C:/Users/human/미정프로젝트/서울시 상권분석서비스(영역-상권).csv', encoding='cp949')

print(f'길단위인구: {len(df_people):,}행')
print(f'영역: {len(df_area):,}행')

길단위인구: 46,184행
영역: 1,650행


In [19]:
# 영역에서 필요한 컬럼만
df_area_slim = df_area[['상권_코드', '자치구_코드_명', '영역_면적']]

# 길단위인구 + 영역 합치기
df_merged = pd.merge(df_people, df_area_slim, on='상권_코드', how='left')

# 연도 추출
df_merged['연도'] = df_merged['기준_년분기_코드'].astype(str).str[:4].astype(int)
df_merged = df_merged[(df_merged['연도'] >= 2019) & (df_merged['연도'] <= 2024)]

print(f'행 수: {len(df_merged):,}개')
print(f'연도 범위: {sorted(df_merged["연도"].unique())}')

행 수: 39,589개
연도 범위: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [20]:
df_people_gu = df_merged.groupby(['자치구_코드_명', '연도']).agg(
    총_유동인구=('총_유동인구_수', 'mean'),
    유동인구_20대=('연령대_20_유동인구_수', 'mean'),
    유동인구_30대=('연령대_30_유동인구_수', 'mean'),
    유동인구_40대=('연령대_40_유동인구_수', 'mean'),
    유동인구_50대=('연령대_50_유동인구_수', 'mean'),
    출근시간_유동인구_06_11=('시간대_06_11_유동인구_수', 'mean'),
    점심시간_유동인구_11_14=('시간대_11_14_유동인구_수', 'mean'),
    저녁시간_유동인구_17_21=('시간대_17_21_유동인구_수', 'mean'),
).reset_index()

df_people_gu = df_people_gu.rename(columns={'자치구_코드_명': '구명'})

print(f'행 수: {len(df_people_gu):,}개')
print(df_people_gu.head(3))

행 수: 150개
    구명    연도         총_유동인구       유동인구_20대       유동인구_30대       유동인구_40대  \
0  강남구  2019  928198.929612  184485.371359  219326.475728  170824.635922   
1  강남구  2020  916064.050971  178571.655340  215503.689320  175009.487864   
2  강남구  2021  871905.388350  163603.487864  204539.686893  171517.216019   

        유동인구_50대  출근시간_유동인구_06_11  점심시간_유동인구_11_14  저녁시간_유동인구_17_21  
0  109888.747573    186583.436893    149304.470874    172608.453883  
1  110989.199029    187169.245146    147216.851942    167626.730583  
2  108914.728155    179233.077670    142024.791262    160101.473301  


In [21]:
files = {
    2019: 'C:/Users/human/미정프로젝트/데이터 전처리/추정매출/서울시_상권분석서비스(추정매출-상권)_2019년.csv',
    2020: 'C:/Users/human/미정프로젝트/데이터 전처리/추정매출/서울시_상권분석서비스(추정매출-상권)_2020년.csv',
    2021: 'C:/Users/human/미정프로젝트/데이터 전처리/추정매출/서울시_상권분석서비스(추정매출-상권)_2021년.csv',
    2022: 'C:/Users/human/미정프로젝트/데이터 전처리/추정매출/서울시_상권분석서비스(추정매출-상권)_2022년.csv',
    2023: 'C:/Users/human/미정프로젝트/데이터 전처리/추정매출/서울시_상권분석서비스(추정매출-상권)_2023년.csv',
    2024: 'C:/Users/human/미정프로젝트/데이터 전처리/추정매출/서울시_상권분석서비스(추정매출-상권)_2024년.csv',
}

dfs = []
for year, path in files.items():
    df_temp = pd.read_csv(path, encoding='cp949')
    df_temp['연도'] = year
    dfs.append(df_temp)
    print(f'{year}년: {len(df_temp):,}행')

df_sales_all = pd.concat(dfs, ignore_index=True)
print(f'\n전체 합계: {len(df_sales_all):,}행')

2019년: 78,427행
2020년: 80,790행
2021년: 89,150행
2022년: 88,834행
2023년: 88,246행
2024년: 87,179행

전체 합계: 512,626행


In [22]:
food_list = ['한식음식점', '중식음식점', '일식음식점', '양식음식점',
             '분식전문점', '치킨전문점', '커피-음료']

df_food_all = df_sales_all[df_sales_all['서비스_업종_코드_명'].isin(food_list)].copy()
df_food_all = pd.merge(df_food_all, df_area_slim, on='상권_코드', how='left')

df_sales_gu = df_food_all.groupby(['자치구_코드_명', '연도', '서비스_업종_코드_명']).agg(
    평균_당월매출=('당월_매출_금액', 'mean'),
    연령대_20_매출=('연령대_20_매출_금액', 'mean'),
    연령대_30_매출=('연령대_30_매출_금액', 'mean'),
    연령대_40_매출=('연령대_40_매출_금액', 'mean'),
    연령대_50_매출=('연령대_50_매출_금액', 'mean'),
    주말_매출_비율=('주말_매출_금액', lambda x: (x.sum() / (df_food_all.loc[x.index, '주중_매출_금액'].sum() + x.sum())).round(4)),
).reset_index()

df_sales_gu = df_sales_gu.rename(columns={
    '자치구_코드_명': '구명',
    '서비스_업종_코드_명': '업종'
})

print(f'행 수: {len(df_sales_gu):,}개')
print(df_sales_gu.head(3))

행 수: 1,050개
    구명    연도     업종       평균_당월매출     연령대_20_매출     연령대_30_매출     연령대_40_매출  \
0  강남구  2019  분식전문점  6.533841e+08  1.279386e+08  1.448395e+08  1.427872e+08   
1  강남구  2019  양식음식점  1.366631e+09  2.688714e+08  3.211270e+08  2.250849e+08   
2  강남구  2019  일식음식점  9.675838e+08  1.774149e+08  2.076301e+08  1.397524e+08   

      연령대_50_매출  주말_매출_비율  
0  8.195965e+07    0.2211  
1  1.425676e+08    0.2955  
2  8.413404e+07    0.1966  


In [23]:
df_closure = pd.read_csv('C:/Users/human/미정프로젝트/새 폴더/폐업률_구별_연도별_업종별.csv', encoding='utf-8-sig')

df_ml = pd.merge(df_sales_gu, df_closure[['구명', '연도', '업종', '폐업률']],
                 on=['구명', '연도', '업종'], how='left')

print(f'행 수: {len(df_ml):,}개')
print(f'결측치: {df_ml["폐업률"].isnull().sum()}개')
print(df_ml[['구명', '연도', '업종', '폐업률']].head())

행 수: 1,050개
결측치: 0개
    구명    연도     업종    폐업률
0  강남구  2019  분식전문점   7.82
1  강남구  2019  양식음식점   9.90
2  강남구  2019  일식음식점  10.23
3  강남구  2019  중식음식점   5.97
4  강남구  2019  치킨전문점  12.26


In [24]:
df_ml = pd.merge(df_ml, df_people_gu, on=['구명', '연도'], how='left')

print(f'행 수: {len(df_ml):,}개')
print(f'결측치: {df_ml.isnull().sum().sum()}개')
print(f'컬럼 수: {len(df_ml.columns)}개')

행 수: 1,050개
결측치: 0개
컬럼 수: 18개


In [25]:
df_subway = pd.read_csv('C:/Users/human/미정프로젝트/서울_상권분석_용_정제_결측치처리_최종_데이터.csv',
                        encoding='utf-8-sig')

df_subway['연도'] = df_subway['사용월'].astype(str).str[:4].astype(int)
df_subway = df_subway[(df_subway['연도'] >= 2019) & (df_subway['연도'] <= 2024)]

승차컬럼 = [col for col in df_subway.columns if '승차인원' in col]
하차컬럼 = [col for col in df_subway.columns if '하차인원' in col]

df_subway['총_승차'] = df_subway[승차컬럼].sum(axis=1)
df_subway['총_하차'] = df_subway[하차컬럼].sum(axis=1)

df_subway_gu = df_subway.groupby(['자치구', '연도']).agg(
    평균_승차=('총_승차', 'mean'),
    평균_하차=('총_하차', 'mean'),
).reset_index()

df_subway_gu = df_subway_gu.rename(columns={'자치구': '구명'})
df_ml = pd.merge(df_ml, df_subway_gu, on=['구명', '연도'], how='left')

print(f'행 수: {len(df_ml):,}개')
print(f'결측치: {df_ml.isnull().sum().sum()}개')

행 수: 1,050개
결측치: 0개


In [26]:
df_work = pd.read_csv('C:/Users/human/미정프로젝트/서울시 상권분석서비스(직장인구-자치구).csv',
                      encoding='cp949')

df_work['연도'] = df_work['기준_년분기_코드'].astype(str).str[:4].astype(int)
df_work = df_work[(df_work['연도'] >= 2019) & (df_work['연도'] <= 2024)]

df_work_gu = df_work.groupby(['자치구_코드_명', '연도']).agg(
    총_직장인구=('총_직장_인구_수', 'mean'),
    직장인구_20대=('연령대_20_직장_인구_수', 'mean'),
    직장인구_30대=('연령대_30_직장_인구_수', 'mean'),
    직장인구_40대=('연령대_40_직장_인구_수', 'mean'),
    직장인구_50대=('연령대_50_직장_인구_수', 'mean'),
).reset_index()

df_work_gu = df_work_gu.rename(columns={'자치구_코드_명': '구명'})
df_ml = pd.merge(df_ml, df_work_gu, on=['구명', '연도'], how='left')

print(f'행 수: {len(df_ml):,}개')
print(f'결측치: {df_ml.isnull().sum().sum()}개')
print(f'컬럼 수: {len(df_ml.columns)}개')

행 수: 1,050개
결측치: 0개
컬럼 수: 25개


In [27]:
print('=== 최종 데이터 확인 ===')
print(f'행 수: {len(df_ml):,}개')
print(f'컬럼 수: {len(df_ml.columns)}개')
print(f'결측치: {df_ml.isnull().sum().sum()}개')
print(f'폐업률 범위: {df_ml["폐업률"].min():.2f}% ~ {df_ml["폐업률"].max():.2f}%')
print()
print('컬럼 목록:')
print(df_ml.columns.tolist())

df_ml.to_csv('C:/Users/human/미정프로젝트/새 폴더/최종_ML용_데이터.csv',
             index=False, encoding='utf-8-sig')
print('\n저장 완료!')

=== 최종 데이터 확인 ===
행 수: 1,050개
컬럼 수: 25개
결측치: 0개
폐업률 범위: 1.98% ~ 23.76%

컬럼 목록:
['구명', '연도', '업종', '평균_당월매출', '연령대_20_매출', '연령대_30_매출', '연령대_40_매출', '연령대_50_매출', '주말_매출_비율', '폐업률', '총_유동인구', '유동인구_20대', '유동인구_30대', '유동인구_40대', '유동인구_50대', '출근시간_유동인구_06_11', '점심시간_유동인구_11_14', '저녁시간_유동인구_17_21', '평균_승차', '평균_하차', '총_직장인구', '직장인구_20대', '직장인구_30대', '직장인구_40대', '직장인구_50대']

저장 완료!


In [28]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 문자형, 목표값, 연도 제외
cat_cols = ['구명', '업종']
target_col = ['폐업률']
except_cols = cat_cols + target_col + ['연도']
num_cols = [col for col in df_ml.columns if col not in except_cols]

# StandardScaler
scaler_std = StandardScaler()
df_standard = df_ml.copy()
df_standard[num_cols] = scaler_std.fit_transform(df_ml[num_cols])

# MinMaxScaler
scaler_mm = MinMaxScaler()
df_minmax = df_ml.copy()
df_minmax[num_cols] = scaler_mm.fit_transform(df_ml[num_cols])

# 저장
df_standard.to_csv('C:/Users/human/미정프로젝트/새 폴더/최종_ML용_데이터_v2_Standard.csv',
                   index=False, encoding='utf-8-sig')
df_minmax.to_csv('C:/Users/human/미정프로젝트/새 폴더/최종_ML용_데이터_v2_MinMax.csv',
                 index=False, encoding='utf-8-sig')

print('저장 완료!')
print(f'연도 확인: {df_standard["연도"].unique()}')
print(f'Standard 매출 범위: {df_standard["평균_당월매출"].min():.2f} ~ {df_standard["평균_당월매출"].max():.2f}')
print(f'MinMax 매출 범위: {df_minmax["평균_당월매출"].min():.2f} ~ {df_minmax["평균_당월매출"].max():.2f}')

저장 완료!
연도 확인: [2019 2020 2021 2022 2023 2024]
Standard 매출 범위: -0.80 ~ 7.34
MinMax 매출 범위: 0.00 ~ 1.00


In [29]:
df_ml['임대료_매출_비율'] = (df_ml['임대료'] / df_ml['평균_당월매출']).round(6)

print(df_ml[['구명', '연도', '업종', '임대료', '평균_당월매출', '임대료_매출_비율', '폐업률']].head(10))

KeyError: '임대료'

In [33]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 문자형, 목표값, 연도 제외
cat_cols = ['구명', '업종']
target_col = ['폐업률']
except_cols = cat_cols + target_col + ['연도']
num_cols = [col for col in df_ml.columns if col not in except_cols]

# StandardScaler
scaler_std = StandardScaler()
df_standard = df_ml.copy()
df_standard[num_cols] = scaler_std.fit_transform(df_ml[num_cols])

# MinMaxScaler
scaler_mm = MinMaxScaler()
df_minmax = df_ml.copy()
df_minmax[num_cols] = scaler_mm.fit_transform(df_ml[num_cols])

# 저장
df_standard.to_csv('C:/Users/human/미정프로젝트/새 폴더/최종_ML용_데이터_v2_Standard.csv',
                   index=False, encoding='utf-8-sig')
df_minmax.to_csv('C:/Users/human/미정프로젝트/새 폴더/최종_ML용_데이터_v2_MinMax.csv',
                 index=False, encoding='utf-8-sig')

print('저장 완료!')
print(f'연도 확인: {df_standard["연도"].unique()}')
print(f'컬럼 수: {len(df_standard.columns)}개')

저장 완료!
연도 확인: [2019 2020 2021 2022 2023 2024]
컬럼 수: 25개
